# 02 - Schedule Primitives

Each section isolates **one** primitive and compares against the fairest possible baseline — a schedule that is identical except for the primitive under study.

| # | Primitive | Transform | Fair baseline |
|---|---|---|---|
| 1 | Reorder | `reorder(i, k, j)` | raw ijk |
| 2 | Tile | `split(j, 32)` + `split(k, 32)` | raw ijk |
| 3 | Unroll | `unroll(j_i)` | same ikj + split(j, 64), **no** unroll |
| 4 | Parallel | `parallel(i)` | raw ijk |
| 5 | Vectorize | `vectorize(j_i)` | same ikj + split(j, 16), **no** vectorize |

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import tvm
import tvm_ffi

from workloads import get_workload

## Setup: baseline, shared buffers, and helpers

In [3]:
mod: tvm.IRModule = get_workload("matmul_256")
N: int     = 256
FLOPS: int = 2 * N**3

sch_baseline: tvm.tir.Schedule    = tvm.tir.Schedule(mod)
built_baseline: tvm.runtime.Module = tvm.build(sch_baseline.mod, target="llvm")
print("Baseline compiled:", built_baseline.is_runnable())

Baseline compiled: True


In [4]:
# Shared timing buffers — reused across every bench() call.
# Correctness checks allocate their own independent copies inside check_correctness().
np.random.seed(42)
A_np: np.ndarray = np.random.randn(N, N).astype("float32")
B_np: np.ndarray = np.random.randn(N, N).astype("float32")
C_np: np.ndarray = np.empty((N, N), dtype="float32")

A_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(A_np)
B_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(B_np)
C_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(C_np)

In [5]:
def check_correctness(
    mod_a: tvm.IRModule,
    mod_b: tvm.IRModule,
    rtol: float = 1e-4,
    atol: float = 1e-4,
) -> bool:
    """
    Parse buffer shapes/dtypes from mod_a["main"].params, generate random
    inputs, run both modules on *separate* copies of every array, and
    compare their output buffers (last parameter by convention).
    """
    func: tvm.tir.PrimFunc = mod_a["main"]
    buffer_map = func.buffer_map

    shapes: list[tuple[int, ...]] = []
    dtypes: list[str] = []
    for param in func.params:
        buf: tvm.tir.Buffer = buffer_map[param]
        shapes.append(tuple(int(s) for s in buf.shape))
        dtypes.append(buf.dtype)

    np.random.seed(0)
    # Allocate completely separate numpy arrays for each run so aliasing
    # or GC cannot corrupt the comparison.
    np_a: list[np.ndarray] = [np.random.randn(*s).astype(d) for s, d in zip(shapes, dtypes)]
    np_b: list[np.ndarray] = [arr.copy() for arr in np_a]

    tvm_a: list[tvm.runtime._tensor.Tensor] = [tvm_ffi.from_dlpack(arr) for arr in np_a]
    tvm_b: list[tvm.runtime._tensor.Tensor] = [tvm_ffi.from_dlpack(arr) for arr in np_b]

    built_a: tvm.runtime.Module = tvm.build(mod_a, target="llvm")
    built_b: tvm.runtime.Module = tvm.build(mod_b, target="llvm")
    built_a(*tvm_a)
    built_b(*tvm_b)

    out_a: np.ndarray = tvm_a[-1].numpy()
    out_b: np.ndarray = tvm_b[-1].numpy()
    return bool(np.allclose(out_a, out_b, rtol=rtol, atol=atol))

In [6]:
def bench(
    built: tvm.runtime.Module,
    *args: tvm.runtime._tensor.Tensor,
    repeat: int = 5,
    number: int = 10,
):
    """Thin wrapper around time_evaluator. Returns result with .mean / .std in seconds."""
    dev: tvm.runtime.Device = tvm.cpu(0)
    return built.time_evaluator("main", dev, repeat=repeat, number=number)(*args)


def report(
    label: str,
    result_variant,
    result_ref,
    ref_label: str = "raw baseline",
) -> None:
    mean_ms: float = result_variant.mean * 1e3
    std_ms:  float = result_variant.std  * 1e3
    gflops:  float = FLOPS / result_variant.mean / 1e9
    speedup: float = result_ref.mean / result_variant.mean
    print(f"{label:<22}  {mean_ms:6.2f} ms ± {std_ms:.2f}  "
          f"{gflops:5.2f} GFLOPS  speedup vs {ref_label}: {speedup:.2f}x")


result_raw = bench(built_baseline, A_tvm, B_tvm, C_tvm)
report("raw baseline", result_raw, result_raw, ref_label="itself")

raw baseline              9.95 ms ± 0.00   3.37 GFLOPS  speedup vs itself: 1.00x


---
## 1 — Loop Reorder

The default **i → j → k** order accesses `B[k, j]` with stride 256 floats as `k` advances — every step is a cache miss.  
Swapping `j` and `k` to **i → k → j** makes the innermost axis stride-1 for both `B` and `C`.

Fair baseline: raw (ijk).

In [7]:
sch_reorder: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_r: tvm.tir.schedule.schedule.BlockRV = sch_reorder.get_block("matmul")
i_r, j_r, k_r = sch_reorder.get_loops(block_r)
sch_reorder.reorder(i_r, k_r, j_r)

print("=== BEFORE (ijk) ===")
sch_baseline.mod.show()
print("\n=== AFTER reorder (ikj) ===")
sch_reorder.mod.show()

=== BEFORE (ijk) ===



=== AFTER reorder (ikj) ===


In [8]:
print("Correct:", check_correctness(sch_baseline.mod, sch_reorder.mod))

built_reorder: tvm.runtime.Module = tvm.build(sch_reorder.mod, target="llvm")
result_reorder = bench(built_reorder, A_tvm, B_tvm, C_tvm)
report("reorder (ikj)", result_reorder, result_raw)

Correct: True
reorder (ikj)             1.60 ms ± 0.01  20.98 GFLOPS  speedup vs raw baseline: 6.22x


---
## 2 — Tiling (`split`)

`split(loop, factors=[outer, inner])` cuts a loop into two nested loops:
```
for j in range(256):   →   for j_o in range(8):     # outer: which tile
                               for j_i in range(32):  # inner: within tile
```

We tile both `j` and `k` with `TILE=32`.  
This groups 32 consecutive `j` values together (stride-1 slice of `B` and `C`) and processes them in chunks of 32 `k` steps, keeping the active `B[k_tile, j_tile]` slice in L1 cache across the inner loops.

Fair baseline: raw ijk.

In [9]:
TILE: int = 32

sch_tile: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_t: tvm.tir.schedule.schedule.BlockRV = sch_tile.get_block("matmul")
i_t, j_t, k_t = sch_tile.get_loops(block_t)

sch_tile.split(j_t, factors=[None, TILE])
sch_tile.split(k_t, factors=[None, TILE])

print("=== BEFORE (ijk) ===")
sch_baseline.mod.show()
print(f"\n=== AFTER tile (j={TILE}, k={TILE}) ===")
sch_tile.mod.show()

=== BEFORE (ijk) ===



=== AFTER tile (j=32, k=32) ===


In [10]:
print("Correct:", check_correctness(sch_baseline.mod, sch_tile.mod))

built_tile: tvm.runtime.Module = tvm.build(sch_tile.mod, target="llvm")
result_tile = bench(built_tile, A_tvm, B_tvm, C_tvm)
report(f"tile (j={TILE}, k={TILE})", result_tile, result_raw)

Correct: True
tile (j=32, k=32)         9.29 ms ± 0.02   3.61 GFLOPS  speedup vs raw baseline: 1.07x


---
## 3 — Unrolling (`sch.unroll`)

`sch.unroll(loop)` forces the compiler to fully inline all loop iterations as independent statements, regardless of LLVM's heuristic unroll threshold (≤ 8 at `-O2`).

Strategy: start from the **ikj** schedule (j innermost), split `j` into `(j_o=4, j_i=64)`, then `unroll(j_i)`.  
Without the annotation, LLVM processes `j_i=64` as a regular loop and fails to handle the init check inside it cleanly — the schedule is actually *slower* than smaller splits.  
With `unroll`, all 64 iterations are inlined as independent statements: no loop-carried dependency, no branch overhead, and the backend can fully pipeline and vectorise across them.

**Fair baseline**: identical `ikj + split(j, 64)` — the *only* difference is the `T.unroll` annotation on `j_i`.

In [11]:
UNROLL_FACTOR: int = 64  # j_o=4 outer iterations, j_i=64 iterations to unroll

# --- Fair baseline: ikj + split(j, 64), NO unroll ---
sch_unroll_base: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_ub: tvm.tir.schedule.schedule.BlockRV = sch_unroll_base.get_block("matmul")
i_ub, j_ub, k_ub = sch_unroll_base.get_loops(block_ub)
sch_unroll_base.reorder(i_ub, k_ub, j_ub)
_, _, j_ub2 = sch_unroll_base.get_loops(block_ub)
sch_unroll_base.split(j_ub2, factors=[None, UNROLL_FACTOR])

# --- Variant: same + unroll(j_i) ---
sch_unroll: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_u: tvm.tir.schedule.schedule.BlockRV = sch_unroll.get_block("matmul")
i_u, j_u, k_u = sch_unroll.get_loops(block_u)
sch_unroll.reorder(i_u, k_u, j_u)
_, _, j_u2 = sch_unroll.get_loops(block_u)
_, j_i = sch_unroll.split(j_u2, factors=[None, UNROLL_FACTOR])
sch_unroll.unroll(j_i)

print(f"=== BEFORE (ikj + split(j={UNROLL_FACTOR}), no unroll) ===")
sch_unroll_base.mod.show()
print(f"\n=== AFTER (+ unroll(j_i={UNROLL_FACTOR})) ===")
sch_unroll.mod.show()

=== BEFORE (ikj + split(j=64), no unroll) ===



=== AFTER (+ unroll(j_i=64)) ===


In [12]:
print("Correct:", check_correctness(sch_unroll_base.mod, sch_unroll.mod))

built_unroll_base: tvm.runtime.Module = tvm.build(sch_unroll_base.mod, target="llvm")
built_unroll:      tvm.runtime.Module = tvm.build(sch_unroll.mod,      target="llvm")

result_unroll_base = bench(built_unroll_base, A_tvm, B_tvm, C_tvm)
result_unroll      = bench(built_unroll,      A_tvm, B_tvm, C_tvm)

report(f"ikj+split(j={UNROLL_FACTOR}) no unroll", result_unroll_base, result_raw)
report(f"ikj+split(j={UNROLL_FACTOR})+unroll",    result_unroll, result_unroll_base,
       ref_label=f"split(j={UNROLL_FACTOR}) no unroll")

Correct: True
ikj+split(j=64) no unroll    5.04 ms ± 0.02   6.66 GFLOPS  speedup vs raw baseline: 1.97x
ikj+split(j=64)+unroll    1.62 ms ± 0.01  20.74 GFLOPS  speedup vs split(j=64) no unroll: 3.11x


---
## 4 — Parallelism (`sch.parallel`)

`sch.parallel(loop)` annotates an axis for multi-threaded execution (OpenMP or TVM's own thread pool).  
We parallelize the outermost `i` loop — each thread computes an independent slice of output rows, so there are **no data races** on `C`.

The expected speedup scales roughly with the number of physical cores.

Fair baseline: raw (ijk).

In [13]:
sch_parallel: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_p: tvm.tir.schedule.schedule.BlockRV = sch_parallel.get_block("matmul")
i_p, _, _ = sch_parallel.get_loops(block_p)
sch_parallel.parallel(i_p)

print("=== BEFORE (ijk) ===")
sch_baseline.mod.show()
print("\n=== AFTER parallel(i) ===")
sch_parallel.mod.show()

=== BEFORE (ijk) ===



=== AFTER parallel(i) ===


In [14]:
print("Correct:", check_correctness(sch_baseline.mod, sch_parallel.mod))

built_parallel: tvm.runtime.Module = tvm.build(sch_parallel.mod, target="llvm")
result_parallel = bench(built_parallel, A_tvm, B_tvm, C_tvm)
report("parallel(i)", result_parallel, result_raw)

Correct: True
parallel(i)               1.24 ms ± 0.01  27.04 GFLOPS  speedup vs raw baseline: 8.02x


[16:49:13] /home/arthur/Documents/Projects/fromsource/d2l-tvm/tvm23/src/runtime/threading_backend.cc:348: Warning: more than two frequencies detected! Forced big_count_ to 16


---
## 5 — Vectorization (`sch.vectorize`)

`sch.vectorize(loop)` tells TVM to generate explicit SIMD vector types for that loop **before** handing off to LLVM.  
Requirements: the annotated loop must be **spatial** (not a reduction) and innermost.

Strategy: start from **ikj** (j innermost), split `j` into `(j_o, j_i=VEC_WIDTH=16)`, then `vectorize(j_i)`.  
`VEC_WIDTH=16` spans two 256-bit AVX2 registers; TVM emits a 16-wide vector type which LLVM lowers to two packed SIMD ops.  
At this width, TVM's explicit codegen is measurably more efficient than LLVM's auto-vectoriser heuristics (~9% gain).

**Fair baseline**: identical `ikj + split(j, 16)` — the *only* difference is the `T.vectorized` annotation on `j_i`.

In [15]:
VEC_WIDTH: int = 16  # 16 × float32 = two 256-bit AVX2 registers

# --- Fair baseline: ikj + split(j, VEC_WIDTH), NO vectorize ---
sch_vec_base: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_vb: tvm.tir.schedule.schedule.BlockRV = sch_vec_base.get_block("matmul")
i_vb, j_vb, k_vb = sch_vec_base.get_loops(block_vb)
sch_vec_base.reorder(i_vb, k_vb, j_vb)
_, _, j_vb2 = sch_vec_base.get_loops(block_vb)  # re-fetch after reorder
sch_vec_base.split(j_vb2, factors=[None, VEC_WIDTH])

# --- Variant: same + vectorize(j_i) ---
sch_vec: tvm.tir.Schedule = tvm.tir.Schedule(mod)
block_v: tvm.tir.schedule.schedule.BlockRV = sch_vec.get_block("matmul")
i_v, j_v, k_v = sch_vec.get_loops(block_v)
sch_vec.reorder(i_v, k_v, j_v)
_, _, j_v2 = sch_vec.get_loops(block_v)
_, j_i = sch_vec.split(j_v2, factors=[None, VEC_WIDTH])
sch_vec.vectorize(j_i)

print(f"=== BEFORE (ikj + split(j={VEC_WIDTH}), no vectorize) ===")
sch_vec_base.mod.show()
print(f"\n=== AFTER (+ vectorize(j_i={VEC_WIDTH})) ===")
sch_vec.mod.show()

=== BEFORE (ikj + split(j=16), no vectorize) ===



=== AFTER (+ vectorize(j_i=16)) ===


In [23]:
print("Correct:", check_correctness(sch_vec_base.mod, sch_vec.mod))

built_vec_base: tvm.runtime.Module = tvm.build(sch_vec_base.mod, target="llvm")
built_vec:      tvm.runtime.Module = tvm.build(sch_vec.mod,      target="llvm")

result_vec_base = bench(built_vec_base, A_tvm, B_tvm, C_tvm)
result_vec      = bench(built_vec,      A_tvm, B_tvm, C_tvm)

report(f"ikj+split(j={VEC_WIDTH}) no vec   ",   result_vec_base, result_raw)
report(f"ikj+split(j={VEC_WIDTH})+vectorize", result_vec, result_vec_base,
       ref_label=f"split(j={VEC_WIDTH}) no vec")

Correct: True
ikj+split(j=16) no vec       1.57 ms ± 0.01  21.31 GFLOPS  speedup vs raw baseline: 6.32x
ikj+split(j=16)+vectorize    1.44 ms ± 0.00  23.23 GFLOPS  speedup vs split(j=16) no vec: 1.09x


---
## Summary

Every primitive follows the same pattern:
```python
sch = tvm.tir.Schedule(mod)
block = sch.get_block("name")
loops = sch.get_loops(block)   # re-fetch after each transform
sch.<primitive>(...)           # annotate / restructure
tvm.build(sch.mod, target="llvm")
```

Measured speedups (fair comparisons, this machine):

| Primitive | Fair baseline | Speedup |
|---|---|---|
| Reorder (ikj) | raw ijk | ~6× |
| Tile (j=32, k=32) | raw ijk | ~1.07× |
| Unroll (j_i=64) | ikj + split(j=64), no annotation | ~3× |
| Parallel (i) | raw ijk | ~7× (scales with core count) |
| Vectorize (j_i=16) | ikj + split(j=16), no annotation | ~1.09× |

Key takeaways:
- **Reorder** wins by far: fixing the memory access pattern is the dominant optimisation.
- **Tile** provides modest cache-blocking gains that compound with other transforms.
- **Unroll** at `j_i=64` exposes 64 independent statements the backend couldn't pipeline from a loop — the gain is not just branch removal but enabling full ILP and vectorisation.
- **Parallel** scales with core count; the annotation is a single line but requires truly independent iterations.
- **Vectorize** at `VEC_WIDTH=16` (two AVX2 registers) consistently beats LLVM's auto-vectoriser by ~9% at this width.

**Next notebook**: combining primitives (reorder + tile + parallel + vectorize) and introducing MetaSchedule to search over the combination space automatically.